In [1]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import IntegerComparator
from math import ceil, log2

from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

In [2]:
def qft_on(circ: QuantumCircuit, qubits):
    n = len(qubits)
    for j in range(n):
        qj = qubits[j]
        circ.h(qj)
        for k in range(j + 1, n):
            qk = qubits[k]
            angle = np.pi / (2 ** (k - j))
            circ.cp(angle, qk, qj)
    # bit reversal
    for j in range(n // 2):
        circ.swap(qubits[j], qubits[n - 1 - j])

In [3]:
def iqft_on(circ: QuantumCircuit, qubits):
    n = len(qubits)
    # undo bit-reversal first
    for j in range(n // 2):
        circ.swap(qubits[j], qubits[n - 1 - j])

    # inverse rotations
    for j in reversed(range(n)):
        qj = qubits[j]
        for k in reversed(range(j + 1, n)):
            qk = qubits[k]
            angle = -np.pi / (2 ** (k - j))  # negative angle
            circ.cp(angle, qk, qj)
        circ.h(qj)

In [4]:
def add_classical_constant_on(circ: QuantumCircuit, qubits, a: int):
    """
    |x> → |x + a (mod 2^n)>
    qubits: MSB → LSB
    """
    qft_on(circ, qubits)
    for j, qj in enumerate(qubits):
        theta = +2 * np.pi * a / (2 ** (j + 1))
        circ.p(theta, qj)
    iqft_on(circ, qubits)

In [5]:
def sub_classical_constant_on(circ: QuantumCircuit, qubits, a: int):
    """
    |x> → |x - a (mod 2^n)>
    """
    qft_on(circ, qubits)
    for j, qj in enumerate(qubits):
        theta = -2 * np.pi * a / (2 ** (j + 1))
        circ.p(theta, qj)
    iqft_on(circ, qubits)

In [6]:
def init_classical_on(circ: QuantumCircuit, reg, value: int):
    """
    reg[0] = MSB, reg[-1] = LSB
    value를 이 순서에 맞게 올려줌.
    """
    n = len(reg)
    bits = format(value, f"0{n}b")  # MSB ... LSB
    for j, b in enumerate(bits):
        if b == "1":
            circ.x(reg[j])

In [7]:
def controlled_sub_classical_on(circ: QuantumCircuit, control, qubits, a: int):
    """
    control=1 일 때만 |x> -> |x - a (mod 2^n)>
    qubits: MSB → LSB
    """
    # 1) QFT
    qft_on(circ, qubits)

    # 2) control이 1일 때만 phase 적용
    for j, qj in enumerate(qubits):
        theta = -2 * np.pi * a / (2 ** (j + 1))
        circ.cp(theta, control, qj)

    # 3) iQFT
    iqft_on(circ, qubits)

In [8]:
def controlled_add_classical_on(circ: QuantumCircuit, control, qubits, a: int):
    """
    control=1 일 때만 |x> -> |x + a (mod 2^n)>
    qubits: MSB → LSB
    """
    if a == 0:
        return  # 0이면 아무 것도 안 해도 됨

    # 1) QFT
    qft_on(circ, qubits)

    # 2) control이 1일 때만 phase 적용
    for j, qj in enumerate(qubits):
        theta = +2 * np.pi * a / (2 ** (j + 1))
        circ.cp(theta, control, qj)

    # 3) iQFT
    iqft_on(circ, qubits)

In [9]:
def build_qtg_with_profit(weights, profits, capacity, theta=np.pi/2):
    """
    QTG 논문 Box 1 구조:
      - register 1: path (아이템 선택 비트들)
      - register 2: capacity (잔여 용량)
      - register 3: profit  (누적 이익)

    For each item m:
      (a) COMP:  cap >= w_m 인 branch에서만 path[m]에 R_y(theta)
      (b) SUB:   path[m]==1 인 branch에서 cap -= w_m
      (c) ADD:   path[m]==1 인 branch에서 profit += p_m

    여기서는 전부 '정수'로 가정.
    weights, profits, capacity 는 이미 정수라고 생각하면 됨.
    (실수일 땐 앞에서 만든 fixed-point 인코딩으로 정수로 바꿔서 넣으면 OK)
    """

    assert len(weights) == len(profits), "weights, profits 길이가 같아야 함"
    n = len(weights)

    # ----- capacity / profit 비트 수 -----
    n_cap = max(1, ceil(log2(capacity + 1)))

    P_max = sum(profits)      # 모든 아이템 다 넣었을 때 최대 profit
    n_profit = max(1, ceil(log2(P_max + 1)))

    # ----- ancilla 개수 파악 (capacity comparator용) -----
    dummy_cmp = IntegerComparator(num_state_qubits=n_cap, value=0, geq=True)
    n_anc = len(dummy_cmp.qregs[2]) if len(dummy_cmp.qregs) > 2 else 0

    # ----- 레지스터 정의 -----
    path   = QuantumRegister(n,        "path")
    cap    = QuantumRegister(n_cap,    "cap")     # MSB..LSB
    profit = QuantumRegister(n_profit, "profit")  # MSB..LSB
    ge     = QuantumRegister(1,        "ge")
    if n_anc > 0:
        wcmp = QuantumRegister(n_anc, "wcmp")
        qc = QuantumCircuit(path, cap, profit, ge, wcmp)
    else:
        wcmp = None
        qc = QuantumCircuit(path, cap, profit, ge)

    # ----- 초기 상태 -----
    # path, profit 은 |0...0>
    # cap = |capacity>
    init_classical_on(qc, cap, capacity)

    # ----- 아이템별 QTG step -----
    for m, (w_m, p_m) in enumerate(zip(weights, profits)):

        # (a) COMP: cap >= w_m ?  → ge 에 기록
        cmp_circ = IntegerComparator(num_state_qubits=n_cap,
                                     value=w_m,
                                     geq=True)

        cap_for_cmp = list(cap)[::-1]  # LSB ... MSB (IntegerComparator convention)
        if n_anc > 0:
            cmp_qubits = cap_for_cmp + [ge[0]] + list(wcmp)
        else:
            cmp_qubits = cap_for_cmp + [ge[0]]

        qc.compose(cmp_circ, cmp_qubits, inplace=True)

        # cap이 충분한 branch에서만 path[m]에 R_y(theta) 적용
        qc.cry(theta, ge[0], path[m])

        # comparator ancilla 정리
        qc.compose(cmp_circ.inverse(), cmp_qubits, inplace=True)

        # (b) SUB: path[m]==1 이면 cap -= w_m
        controlled_sub_classical_on(qc, path[m], cap, w_m)

        # (c) ADD: path[m]==1 이면 profit += p_m
        controlled_add_classical_on(qc, path[m], profit, p_m)

    return qc


In [10]:
def print_full_statevector_clean(sv, threshold=1e-6, forward=False):
    """
    범용 Statevector 출력 함수.
    - forward=False  : MSB->LSB (사람 기준, Qiskit 기준)
    - forward=True   : LSB->MSB (QFT/ADD/SUB 디버깅용)
                      + index도 LSB 기준 숫자 증가 순으로 정렬
    
    """
    if not isinstance(sv, Statevector):
        sv = Statevector(sv)
    
    amps = sv.data
    n = sv.num_qubits

    print(f"Full Cleaned Statevector (threshold={threshold}, forward={forward})")
    print("-" * 60)

    # 1) 인덱스 리스트 준비
    indices = list(range(len(amps)))

    if forward:
        # ★ forward=True 일 때는 index를 비트 뒤집은 기준으로 정렬해야 함
        def bit_reverse(i):
            raw = format(i, f'0{n}b')  # q2 q1 q0
            rev = raw[::-1]           # q0 q1 q2
            return int(rev, 2)        # rev를 다시 숫자로 해석

        indices.sort(key=lambda i: bit_reverse(i))
    else:
        # 기본은 기존 인덱스 순서 그대로 출력
        pass

    # 2) 출력
    for i in indices:
        amp = amps[i]
        raw = format(i, f'0{n}b')      # q2 q1 q0

        if forward:
            bitstring = raw[::-1]      # LSB->MSB
        else:
            bitstring = raw            # MSB->LSB

        if abs(amp) < threshold:
            # print(f"|{bitstring}> : 0")
            pass
        else:
            print(f"|{bitstring}> : {amp.real:+.6f}{amp.imag:+.6f}j")

    print("-" * 60)

In [11]:
weights  = [3, 2, 4]
profits  = [6, 2, 5]   # 예시
capacity = 5

qc = build_qtg_with_profit(weights, profits, capacity, theta=np.pi/2)
print(qc.draw())

                                ┌─────────┐                                  »
  path_0: ──────────────────────┤ Ry(π/2) ├──────────────────────────────────»
                                └────┬────┘                                  »
  path_1: ───────────────────────────┼───────────────────────────────────────»
                                     │                                       »
  path_2: ───────────────────────────┼───────────────────────────────────────»
          ┌───┐┌──────┐              │                   ┌─────────┐  ┌───┐  »
   cap_0: ┤ X ├┤2     ├──────────────┼───────────────────┤2        ├──┤ H ├──»
          └───┘│      │              │                   │         │  └───┘  »
   cap_1: ─────┤1     ├──────────────┼───────────────────┤1        ├─────────»
          ┌───┐│      │              │                   │         │         »
   cap_2: ┤ X ├┤0     ├──────────────┼───────────────────┤0        ├─────────»
          ├───┤│      │              │              

In [12]:
sv = Statevector.from_instruction(qc)
print_full_statevector_clean(sv, threshold=1e-6, forward=True)

Full Cleaned Statevector (threshold=1e-06, forward=True)
------------------------------------------------------------
|0001010000000> : +0.353553+0.000000j
|0010010101000> : +0.353553+0.000000j
|0100110010000> : +0.500000+0.000000j
|1000100110000> : +0.500000-0.000000j
|1100001000000> : +0.500000-0.000000j
------------------------------------------------------------
